# Final pair-wise goodness-of-fit distributions

For each modelled pair, the **final verified distribution** (the one the fallback cascade
selected) is fitted to that pair's headways, and its fit is reported and plotted:
histogram + KDE + fitted PDF, and empirical CDF + fitted CDF, with KS and Anderson–Darling
statistics.

* The final distribution per pair is read from `06_model_fit_verification.xlsx` (falls back
  to a built-in mapping if that file is absent).
* Fits use a free `loc` (best marginal shape — this is the distribution figure, distinct from
  the covariate-conditioned model verified by Cox–Snell in notebook 06).
* **Guaranteed outputs** (no matplotlib): a GoF summary table, the plot data, and native
  Excel charts, all in the `Tables` workbook. Publication PNGs are then attempted with
  matplotlib and saved to `Graphics`; if matplotlib is blocked, that step is skipped
  gracefully and the Excel outputs still stand.

In [1]:
# --- Cell 1: Imports and paths (no matplotlib at top) ---
import os
import numpy as np
import pandas as pd
from scipy import stats

BASE     = r"D:\Headway"
DATA     = os.path.join(BASE, "data3.xlsx")
TABLES   = os.path.join(BASE, "Tables")
GRAPHICS = os.path.join(BASE, "Graphics")
MODEL06  = os.path.join(TABLES, "06_model_fit_verification.xlsx")
os.makedirs(TABLES, exist_ok=True); os.makedirs(GRAPHICS, exist_ok=True)

OUTCOME = "Time_Headway"

# fallback mapping (used only if 06_model_fit_verification.xlsx is not found)
FINAL_DIST_FALLBACK = {
    "BTW_following_4W": "weibull_min", "BTW_following_MT_3W": "gamma",
    "BTW_following_NMT_3W": "gengamma", "PR_following_MT_3W": "weibull_min",
    "BTW_following_MT_2W": "gengamma", "PR_following_NMT_3W": "weibull_min",
    "PR_following_4W": "gengamma", "BTW_following_NMT_2W": "weibull_min",
}
PRETTY = {"weibull_min": "Weibull", "gamma": "Gamma", "gengamma": "Generalised gamma",
          "lognorm": "Lognormal", "invgauss": "Inverse Gaussian", "fisk": "Log-logistic",
          "pearson3": "Pearson III", "rayleigh": "Rayleigh", "expon": "Exponential"}

In [2]:
# --- Cell 2: Load data + final distribution per pair (from notebook 06) ---
df = pd.read_excel(DATA)
try:
    g06 = pd.read_excel(MODEL06, sheet_name="Model_fit_GoF")
    FINAL_DIST = dict(zip(g06["Pair"], g06["Distribution"]))
    print("Final distributions read from 06_model_fit_verification.xlsx")
except Exception as e:
    FINAL_DIST = dict(FINAL_DIST_FALLBACK)
    print("06 workbook not found - using built-in mapping. (", type(e).__name__, ")")

# order pairs by sample size (descending)
order = (df[df["Pair"].isin(FINAL_DIST)].groupby("Pair")[OUTCOME].size()
           .sort_values(ascending=False).index.tolist())
for p in order:
    print(f"  {p:24s} -> {FINAL_DIST[p]}")

Final distributions read from 06_model_fit_verification.xlsx
  BTW_following_4W         -> weibull_min
  BTW_following_MT_3W      -> gamma
  BTW_following_NMT_3W     -> gengamma
  PR_following_MT_3W       -> weibull_min
  BTW_following_MT_2W      -> gengamma
  PR_following_NMT_3W      -> weibull_min
  PR_following_4W          -> gengamma
  BTW_following_NMT_2W     -> weibull_min


In [3]:
# --- Cell 3: Fit final distribution + GoF + plot data per pair ---
def ad_statistic(Fsorted):
    F = np.clip(np.sort(Fsorted), 1e-12, 1 - 1e-12); n = len(F); i = np.arange(1, n+1)
    return -n - np.sum((2*i - 1)/n * (np.log(F) + np.log(1 - F[::-1])))

gof_rows, PLOT = [], {}
for pair in order:
    t = df.loc[df["Pair"] == pair, OUTCOME].dropna().values
    n = len(t); dn = FINAL_DIST[pair]; dist = getattr(stats, dn)
    params = dist.fit(t)                                # free loc: best marginal shape
    ll = float(np.sum(dist.logpdf(t, *params))); k = len(params)
    ksD, ksP = stats.kstest(t, dn, args=params)
    A2 = ad_statistic(dist.cdf(t, *params))
    shapes = (dist.shapes.split(",") if dist.shapes else [])
    labels = [s.strip() for s in shapes] + ["loc", "scale"]
    pstr = ", ".join(f"{l}={v:.4f}" for l, v in zip(labels, params))
    gof_rows.append(dict(Pair=pair, N=n, Distribution=PRETTY.get(dn, dn),
                         KS_D=round(ksD, 4), KS_p=round(ksP, 4), AD_A2=round(A2, 3),
                         logL=round(ll, 2), AIC=round(2*k - 2*ll, 2),
                         KS_verdict="fits (p>=0.05)" if ksP >= 0.05 else "REJECTED",
                         Params=pstr))
    # plot data
    xg = np.linspace(t.min(), t.max(), 200)
    nb = int(min(25, max(6, round(np.sqrt(n)))))
    dens, edges = np.histogram(t, bins=nb, density=True); mid = (edges[:-1] + edges[1:]) / 2
    kde = stats.gaussian_kde(t)(xg)
    xs = np.sort(t); ecdf = np.arange(1, n+1) / n
    PLOT[pair] = dict(dn=dn, params=params, t=t, xg=xg, pdf=dist.pdf(xg, *params),
                      cdf=dist.cdf(xg, *params), mid=mid, dens=dens, kde=kde,
                      ecdf_x=xs, ecdf_y=ecdf, ksD=ksD, ksP=ksP)
gof_summary = pd.DataFrame(gof_rows)
gof_summary

,Pair,N,Distribution,KS_D,KS_p,AD_A2,logL,AIC,KS_verdict,Params
0,BTW_following_4W,250,Weibull,0.0344,0.9187,0.246,-308.22,622.44,fits (p>=0.05),"c=2.4049, loc=0.3652, scale=2.1645"
1,BTW_following_MT_3W,186,Gamma,0.0375,0.9474,0.282,-218.11,442.21,fits (p>=0.05),"a=3.0859, loc=0.3196, scale=0.5002"
2,BTW_following_NMT_3W,119,Generalised gamma,0.0560,0.8286,0.393,-142.71,293.42,fits (p>=0.05),"a=0.3686, c=3.0339, loc=0.5301, scale=2.5862"
3,PR_following_MT_3W,104,Weibull,0.0746,0.5825,0.437,-137.32,280.64,fits (p>=0.05),"c=3.5958, loc=-0.0591, scale=3.2413"
4,BTW_following_MT_2W,73,Generalised gamma,0.0713,0.8259,0.588,-77.40,162.79,fits (p>=0.05),"a=0.4376, c=2.2363, loc=0.6000, scale=2.1461"
5,PR_following_NMT_3W,49,Weibull,0.0980,0.6981,0.379,-68.32,142.64,fits (p>=0.05),"c=3.5558, loc=-0.3896, scale=3.4471"
6,PR_following_4W,43,Generalised gamma,0.1164,0.5656,0.417,-60.76,129.53,fits (p>=0.05),"a=0.1499, c=8.7833, loc=0.8197, scale=3.8736"
7,BTW_following_NMT_2W,41,Weibull,0.0733,0.9688,0.259,-44.04,94.08,fits (p>=0.05),"c=1.4571, loc=0.5148, scale=1.3001"


In [4]:
# --- Cell 4: Save GoF table + plot data + native Excel charts (matplotlib-free) ---
from openpyxl import load_workbook
from openpyxl.chart import ScatterChart, Reference, Series

out_path = os.path.join(TABLES, "07_pairwise_gof_distributions.xlsx")
with pd.ExcelWriter(out_path, engine="openpyxl") as xl:
    gof_summary.to_excel(xl, sheet_name="GoF_summary", index=False)

wb = load_workbook(out_path)
for pair in order:
    d = PLOT[pair]; sh = ("fit_" + pair)[:31].replace("/", "_")
    ws = wb.create_sheet(sh)
    ws["A1"], ws["B1"], ws["C1"] = "hist_mid", "hist_density", "kde_x"
    ws["D1"], ws["E1"] = "x", "pdf"
    ws["G1"], ws["H1"], ws["J1"], ws["K1"] = "ecdf_x", "ecdf_y", "x", "cdf"
    for i in range(len(d["mid"])):
        ws.cell(i+2, 1, float(d["mid"][i])); ws.cell(i+2, 2, float(d["dens"][i]))
    for i in range(len(d["xg"])):
        ws.cell(i+2, 3, float(d["xg"][i])); ws.cell(i+2, 4, float(d["xg"][i]))
        ws.cell(i+2, 5, float(d["pdf"][i])); ws.cell(i+2, 10, float(d["xg"][i]))
        ws.cell(i+2, 11, float(d["cdf"][i]))
    for i in range(len(d["ecdf_x"])):
        ws.cell(i+2, 7, float(d["ecdf_x"][i])); ws.cell(i+2, 8, float(d["ecdf_y"][i]))
    # density chart: histogram points + fitted PDF line
    ch1 = ScatterChart(); ch1.title = f"{pair}: density"; ch1.x_axis.title="headway (s)"; ch1.y_axis.title="density"
    ch1.x_axis.delete = False; ch1.y_axis.delete = False
    s_h = Series(Reference(ws,min_col=2,min_row=1,max_row=len(d["mid"])+1),
                 Reference(ws,min_col=1,min_row=2,max_row=len(d["mid"])+1), title_from_data=True)
    s_h.marker.symbol="circle"; s_h.graphicalProperties.line.noFill=True
    s_p = Series(Reference(ws,min_col=5,min_row=1,max_row=len(d["xg"])+1),
                 Reference(ws,min_col=4,min_row=2,max_row=len(d["xg"])+1), title_from_data=True); s_p.smooth=True
    ch1.series.append(s_h); ch1.series.append(s_p); ch1.height, ch1.width = 8, 12
    ws.add_chart(ch1, "M2")
    # CDF chart: ECDF + fitted CDF
    ch2 = ScatterChart(); ch2.title = f"{pair}: CDF (KS p={d['ksP']:.3f})"; ch2.x_axis.title="headway (s)"; ch2.y_axis.title="F(x)"
    ch2.x_axis.delete = False; ch2.y_axis.delete = False
    s_e = Series(Reference(ws,min_col=8,min_row=1,max_row=len(d["ecdf_x"])+1),
                 Reference(ws,min_col=7,min_row=2,max_row=len(d["ecdf_x"])+1), title_from_data=True)
    s_e.marker.symbol="circle"; s_e.graphicalProperties.line.noFill=True
    s_c = Series(Reference(ws,min_col=11,min_row=1,max_row=len(d["xg"])+1),
                 Reference(ws,min_col=10,min_row=2,max_row=len(d["xg"])+1), title_from_data=True); s_c.smooth=True
    ch2.series.append(s_e); ch2.series.append(s_c); ch2.height, ch2.width = 8, 12
    ws.add_chart(ch2, "M20")
wb.save(out_path)
print("Saved (guaranteed, matplotlib-free):", out_path)

Saved (guaranteed, matplotlib-free): D:\Headway\Tables\07_pairwise_gof_distributions.xlsx


In [5]:
# --- Cell 5: Publication PNGs via matplotlib (skipped gracefully if blocked) ---
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.size": 10, "axes.grid": True, "grid.alpha": 0.3,
                         "figure.dpi": 300, "savefig.bbox": "tight"})

    # per-pair 2-panel figures
    for pair in order:
        d = PLOT[pair]; dn = PRETTY.get(d["dn"], d["dn"]); n = len(d["t"])
        fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
        ax[0].hist(d["t"], bins=len(d["mid"]), density=True, color="#c9d6e5",
                   edgecolor="white", label="observed")
        ax[0].plot(d["xg"], d["kde"], "--", color="#555", lw=1.2, label="KDE")
        ax[0].plot(d["xg"], d["pdf"], "-", color="#c0392b", lw=1.8, label=f"{dn} fit")
        ax[0].set_xlabel("Time headway (s)"); ax[0].set_ylabel("Density"); ax[0].legend(frameon=False)
        ax[1].step(d["ecdf_x"], d["ecdf_y"], where="post", color="#2c3e50", lw=1.2, label="Empirical")
        ax[1].plot(d["xg"], d["cdf"], "-", color="#c0392b", lw=1.8, label=f"{dn} fit")
        ax[1].set_xlabel("Time headway (s)"); ax[1].set_ylabel("F(x)")
        ax[1].annotate(f"KS D={d['ksD']:.3f}, p={d['ksP']:.3f}", xy=(0.05, 0.9),
                       xycoords="axes fraction", fontsize=8)
        ax[1].legend(frameon=False, loc="lower right")
        fig.suptitle(f"{pair}   (n={n}, {dn})", fontsize=11)
        fp = os.path.join(GRAPHICS, f"gof_{pair}.png".replace("/", "_"))
        fig.savefig(fp); plt.close(fig)

    # combined grid of PDF panels
    ncol = 2; nrow = int(np.ceil(len(order)/ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(9, 2.6*nrow))
    for axk, pair in zip(np.ravel(axes), order):
        d = PLOT[pair]
        axk.hist(d["t"], bins=len(d["mid"]), density=True, color="#c9d6e5", edgecolor="white")
        axk.plot(d["xg"], d["pdf"], "-", color="#c0392b", lw=1.6)
        axk.set_title(f"{pair}\n{PRETTY.get(d['dn'],d['dn'])}, KS p={d['ksP']:.2f}", fontsize=8)
        axk.tick_params(labelsize=7)
    for axk in np.ravel(axes)[len(order):]: axk.axis("off")
    fig.supxlabel("Time headway (s)"); fig.supylabel("Density"); fig.tight_layout()
    grid_fp = os.path.join(GRAPHICS, "gof_all_pairs_grid.png")
    fig.savefig(grid_fp); plt.close(fig)
    print("Saved publication PNGs to:", GRAPHICS)
except Exception as e:
    print("matplotlib unavailable (", type(e).__name__, ") - skipped PNGs.")
    print("Use the native Excel charts in 07_pairwise_gof_distributions.xlsx, or the exported")
    print("plot-data columns to render figures in another tool.")

Saved publication PNGs to: D:\Headway\Graphics
